# Plan-graph Vendi with learned GIN embeddings (cosine kernel)

## 0 · Setup

In [ ]:
import sys, json, hashlib, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
for _p in (str(ROOT), str(Path.cwd())):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from config import WORKLOADS_DIR
from qpp.models import TrinoGraphGINPlanEncoder
from qpp.config import SEED           # same base seed as the QPP notebooks
from workloads import ANALYSIS_RESULTS_DIR
from utils.workload_analysis import dags_from_plan_bundle, vendi_score_from_similarity

WORKLOAD_ROOT = WORKLOADS_DIR
RESULTS_DIR = ANALYSIS_RESULTS_DIR   # reads workload_metrics.csv; writes gin_vendi/ where tables.ipynb expects it
CACHE_DIR = RESULTS_DIR / "gin_vendi"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

REF = "tpcds_ref"
BASELINES = [
    "tpcds_bootstrapping_lcm",
    "tpcds_e2etune",
    "tpcds_sql_factory",
    "tpcds_sqlbarber",
    "tpcds_sqlstorm",
]
OURS = ["tpcds_diversql_no_feedback", "tpcds_diversql"]

ROWS = [REF] + BASELINES + OURS

PRETTY = {
    "tpcds_ref": "TPC-DS (1000)",
    "tpcds_bootstrapping_lcm": "Bootstrapping-LCM",
    "tpcds_e2etune": "E2ETune",
    "tpcds_sql_factory": "SQL-Factory",
    "tpcds_sqlbarber": "SQLBarber",
    "tpcds_sqlstorm": "SQLStorm",
    "tpcds_diversql_no_feedback": "DiverSQL (W/O FB)",
    "tpcds_diversql": "DiverSQL (W/ FB)",
}

_INV = {v:k for k,v in PRETTY.items()}

def show(tbl, fmt="{:.3f}", **kw):
    disp = tbl.rename(index=PRETTY)
    def hl(row):
        key = _INV.get(row.name, row.name)
        if key == REF:  return ["background-color:#f3f4f6;font-style:italic"]*len(row)
        if key in OURS: return ["background-color:#fff1e6;font-weight:600"]*len(row)
        return [""]*len(row)
    sty = disp.style.apply(hl, axis=1)
    return sty.format(fmt, **kw) if fmt else sty


## 1 · Load the existing compiled plans

`plans.json` is the cached EXPLAIN bundle written by `analysis.capture_query_plans`. `dags_from_plan_bundle`
keeps only `status == ok` plans, the same filter Vendi (κ) uses. Nothing here talks to Trino.

In [ ]:
dags_by_workload = {}
plan_meta = []
for w in ROWS:
    bundle = json.loads((WORKLOAD_ROOT / w / "plans.json").read_text(encoding="utf-8"))
    dags_by_workload[w] = dags_from_plan_bundle(bundle)
    plan_meta.append({"workload": w, "plans_ok": len(dags_by_workload[w]),
                      "query_count": bundle.get("query_count"), "captured_at_utc": bundle.get("captured_at_utc")})

df_plans = pd.DataFrame(plan_meta).set_index("workload")
display(show(df_plans, None))

# pooled training set: every plan of every workload, in a fixed order
POOLED = [(w, q) for w in ROWS for q in dags_by_workload[w]]
POOLED_PLANS = [dags_by_workload[w][q] for w, q in POOLED]
print(f"pooled training plans: {len(POOLED_PLANS)}")


## 2 · Training hyperparameters

Edit here. Defaults are the QPP evaluation's GIN config. `FORCE_RETRAIN = True` refits even when a cache
for these exact settings exists.

In [ ]:
HPARAMS = {
    # architecture
    "hidden":       128,     # node-state width
    "n_layers":     3,       # GINE message-passing layers
    "emb_dim":      256,     # exported embedding dim (None -> (n_layers+1)*hidden readout, no projection)
    "dropout":      0.1,
    # self-supervised training
    "mask_rate":    0.15,    # fraction of nodes masked for masked-node modelling
    "epochs":       300,
    "batch_size":   256,
    "lr":           1e-3,
    "weight_decay": 1e-4,
    "clip_grad":    5.0,     # grad-norm clip
    "w_num":        1.0,     # loss weight: masked numeric-estimate reconstruction
    "w_graph":      1.0,     # loss weight: graph-property regression
    "seed":         SEED,    # 42
}
FORCE_RETRAIN = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# vocabulary from the pooled plans themselves (never the QPP closed-world fixed_vocab)
ENCODER_KWARGS = {**HPARAMS, "fixed_vocab": None}
print(f"device={DEVICE}")


## 3 · Train once on the pooled plans, embed every workload

One `fit_from_train_plans` on all pooled plans, then a frozen `encode_plan` for every plan. Encoder and
embeddings are cached as `results/gin_vendi/{gin,emb}_pooled_<fingerprint>`. The fingerprint covers
`ENCODER_KWARGS` and the plan set, so stale embeddings can't be picked up silently.

In [ ]:
def _fingerprint(kwargs):
    kw = {k: v for k, v in kwargs.items() if k not in ("device", "verbose")}
    plans_sig = [(w, q, len(d["nodes"]), len(d["edges"])) for (w, q), d in zip(POOLED, POOLED_PLANS)]
    blob = json.dumps({"kwargs": kw, "plans": plans_sig}, sort_keys=True, default=str)
    return hashlib.sha256(blob.encode()).hexdigest()[:16]

FP = _fingerprint(ENCODER_KWARGS)
enc_path, emb_path = CACHE_DIR / f"gin_pooled_{FP}.pt", CACHE_DIR / f"emb_pooled_{FP}.npz"

if emb_path.exists() and not FORCE_RETRAIN:
    Z = np.load(emb_path, allow_pickle=False)["Z"]
    print(f"loaded cached embeddings {emb_path.name}")
else:
    t0 = time.time()
    enc = TrinoGraphGINPlanEncoder.fit_from_train_plans(
        POOLED_PLANS, **ENCODER_KWARGS, device=DEVICE, verbose=True)
    print(f"trained in {(time.time() - t0) / 60:.1f} min, dim={enc.dim()}, vocab={len(enc._op2idx)}")
    Z = enc.encode_plans(POOLED_PLANS)          # frozen, unscaled -- as make_plan_embeddings_by_run returns
    enc.save(str(enc_path))
    np.savez(emb_path, Z=Z, fingerprint=FP,
             workload=np.array([w for w, _ in POOLED]), query=np.array([q for _, q in POOLED]))
    print(f"cached -> {emb_path.name}")
print(f"embeddings: {Z.shape}")


## 4 · Plan-graph Vendi with a cosine kernel

The embeddings are standardised with a `StandardScaler` fitted once on the pooled set, so every workload is
measured in one shared space. Then for each workload `K = Ẑ Ẑᵀ`, where `Ẑ` is the L2-normalised rows
(`K_ii = 1`, PSD), and `K` goes through `vendi_score_from_similarity`. Vendi (κ) is read from `results/*.csv`
exactly as `tables_2_3_4.ipynb` reads it, for comparison. Mean off-diagonal cosine is a diagnostic: values near
1 would mean the space has collapsed.

In [ ]:
COSINE_COL = "GIN cosine, pooled vocab, std." 

def _unit(Z):
    norms = np.linalg.norm(Z, axis=1, keepdims=True)
    assert (norms > 0).all(), "zero-norm embedding: cosine undefined"
    return Z / norms

def cosine_vendi(Z):
    Zn = _unit(Z)
    return vendi_score_from_similarity(Zn @ Zn.T)

def mean_offdiag_cosine(Z):
    Zn = _unit(Z); K = Zn @ Zn.T; n = len(K)
    return float((K.sum() - np.trace(K)) / (n * (n - 1)))

Z_std = StandardScaler().fit_transform(Z)
_wl = np.array([w for w, _ in POOLED])

_df = None
for _p in sorted(RESULTS_DIR.glob("*.csv"), key=lambda p: p.stat().st_mtime):
    _d = pd.read_csv(_p)
    if "workload" not in _d.columns:
        continue
    _d = _d.set_index("workload")
    _df = _d if _df is None else _d.combine_first(_df)

table = pd.DataFrame({
    "Plans":               df_plans["plans_ok"],
    "Vendi (kappa)":       _df.loc[ROWS, "plan_graph_vendi_score"].astype(float),
    COSINE_COL:            [cosine_vendi(Z_std[_wl == w]) for w in ROWS],
    "Mean cosine (diag.)": [mean_offdiag_cosine(Z_std[_wl == w]) for w in ROWS],
}, index=ROWS)

competitors = [w for w in ROWS if w != REF]
for c in ["Vendi (kappa)", COSINE_COL]:
    table.loc[competitors, f"Rank: {c}"] = table.loc[competitors, c].rank(ascending=False, method="min")

show(table, None).format({"Plans": "{:.0f}", "Vendi (kappa)": "{:.1f}", COSINE_COL: "{:.1f}",
                          "Mean cosine (diag.)": "{:.3f}",
                          f"Rank: Vendi (kappa)": "{:.0f}", f"Rank: {COSINE_COL}": "{:.0f}"}, na_rep="—")


In [ ]:
table.to_csv(CACHE_DIR / "plan_vendi_gin_cosine.csv")
(CACHE_DIR / "plan_vendi_gin_cosine_hparams.json").write_text(json.dumps(
    {"fingerprint": FP, "embeddings": emb_path.name, "encoder_kwargs": ENCODER_KWARGS}, indent=2, default=str))
print(f"wrote plan_vendi_gin_cosine.csv  (fingerprint {FP})")

s = table.loc[competitors, COSINE_COL].sort_values(ascending=False)
print(f"\nVendi (cosine) ranking: " + "  >  ".join(f"{PRETTY[w]} ({v:.1f})" for w, v in s.items()))
rho = table.loc[competitors, ["Vendi (kappa)", COSINE_COL]].corr(method="spearman").iloc[0, 1]
print(f"Spearman rank correlation with Vendi (kappa), 7 generators: rho = {rho:+.3f}")
